# ***Statistical Learning on CASdatasets-$\texttt{fremotorclaim}$***

## ***Modeling Claim Count $N$***
In the following `code` cells, we compress the data frame, leaving only the explanatory risk factors and total claim counts for each risk profile. We then fit the Poisson GLM with and without `year` as a risk factor. The **likelihood-ratio test** finds that `year` is a statistically significant risk factor ($\Lambda = 353.4$, $q=2$, $p \approx 1.8\times10^{-77}$). 

Further, we compute the ratio of expected claim counts across years holding other risk factors fixed. With `year = 7` as the baseline, years 8 and 9 have expected claim counts approximately 1.039 and 1.085 times that of year 7, respectively.

Finally, we split the data frame into training (years 7–8) and testing (year 9) datasets, with `year` encoded as numeric so the fitted  trend can be extrapolated to `year = 9`.

In [25]:
load("euMTPL.rda")
df_eu = euMTPL
df_eu$year = factor(df_eu$year)
df_eu$num_cl = df_eu$num_nc + df_eu$num_cg + df_eu$num_fcg + df_eu$num_cd # These are counts with different settlement methods
df_eu = df_eu[, c(1,3,4,5,6,8,9,10,11,12,14,16,18,20)]

In [26]:
dim(df_eu); head(df_eu); summary(df_eu)

[1] 2373197      14

,policy_id,fuel_type,year,vehicle_category,vehicle_use,horsepower,gender,age,exposure,cost_nc,cost_cg,cost_fcg,cost_cd,num_cl
,<int>,<fct>,<fct>,<fct>,<fct>,<int>,<fct>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
1,1,B,7,1,1,14,M,77,0.48767123,0,0,0,0,0
2,2,B,7,1,1,12,M,40,0.01917808,0,0,0,0,0
3,4,B,7,1,1,14,M,75,0.03287671,0,0,0,0,0
4,5,B,7,1,1,13,M,48,0.04383562,0,0,0,0,0
5,6,B,7,1,1,12,F,54,0.04657534,0,0,0,0,0
6,8,B,7,1,1,12,F,34,0.07671233,0,0,0,0,0


   policy_id         fuel_type       year       vehicle_category vehicle_use 
 Min.   :      1   B      :1382612   7:788932   1:2360481        0 :  12670  
 1st Qu.: 641820   D      : 531255   8:789367   8:  12716        1 :2359462  
 Median :1280437   G      : 383832   9:794898                    4 :    145  
 Mean   :1290070   T      :  29637                               5 :    518  
 3rd Qu.:1939574   P      :  19289                               26:    402  
 Max.   :2595214   S      :  15131                                           
                   (Other):  11441                                           
   horsepower     gender           age            exposure     
 Min.   :  0.00   F: 893348   Min.   : 18.00   Min.   :0.0010  
 1st Qu.: 14.00   M:1479849   1st Qu.: 37.00   1st Qu.:0.3005  
 Median : 16.00               Median : 46.00   Median :0.6822  
 Mean   : 16.52               Mean   : 48.17   Mean   :0.6266  
 3rd Qu.: 19.00               3rd Qu.: 59.00   3rd Qu.:1

In [29]:
df_eu[df_eu$num_cl>0, ]

,policy_id,fuel_type,year,vehicle_category,vehicle_use,horsepower,gender,age,exposure,cost_nc,cost_cg,cost_fcg,cost_cd,num_cl
,<int>,<fct>,<fct>,<fct>,<fct>,<int>,<fct>,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<int>
41,44,B,7,1,1,12,F,31,0.42739726,0.00,1584.94,1800,0,2
173,182,B,7,1,1,18,M,45,1.00000000,0.00,0.00,0,1800,1
175,184,D,7,1,1,18,M,61,0.48493151,1632.00,0.00,0,0,1
187,196,B,7,1,1,18,F,74,0.56438356,0.00,0.00,0,1800,1
188,197,B,7,1,1,14,M,37,1.00000000,303.59,0.00,0,0,1
194,203,B,7,1,1,15,M,53,1.00000000,6231.23,0.00,0,0,1
206,215,D,7,1,1,19,F,60,0.21643836,60.49,0.00,0,2000,2
382,396,B,7,1,1,15,M,60,0.16164384,0.00,935.36,2000,0,2
446,463,B,7,1,1,12,M,29,0.22465753,0.00,0.00,0,2300,1


In [14]:
# poisson.fit.full = glm(num_cl ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)
# poisson.fit.reduced = glm(num_cl ~ fuel_type + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_eu)

In [15]:
# saveRDS(poisson.fit.full, "fit.full.rds")
# saveRDS(poisson.fit.reduced, "fit.reduced.rds")
# file.exists("fit.full.rds")
# file.exists("fit.reduced.rds")

In [16]:
fit.full = readRDS("fit.full.rds")
fit.reduced = readRDS("fit.reduced.rds")

In [17]:
anova(fit.full, fit.reduced, test = "LRT") 

,Resid. Df,Resid. Dev,Df,Deviance,Pr(>Chi)
,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,2373179,1624298,NA,NA,NA
2,2373181,1624652,-2,-353.4045,1.816285e-77


In [18]:
coefs = summary(fit.full)$coefficients
year_coefs = coefs[grep("year", rownames(coefs)), ]
contrasts(df_eu$year); exp(year_coefs[, "Estimate"]) #contrasts() shows that year 7 is the baseline.

,8,9
7,0,0
8,1,0
9,0,1


year8    year9 
1.038986 1.084528

In [19]:
df_train = df_eu[df_eu$year %in% c("7", "8"), ]
df_test = df_eu[df_eu$year == "9", ]

df_train$year = as.numeric(as.character(df_train$year))
df_test$year = as.numeric(as.character(df_test$year))

fit.count.train = glm(num_cl ~ fuel_type + year + vehicle_category + vehicle_use + horsepower + gender + age + offset(log(exposure)), family = poisson, df_train)
pred.count.test = predict(fit.count.train, newdata = df_test, type = "response")

### ***Model Significance & Goodness-of-Fit Assessment***
The following `code` provide the results of **Likelihood-Ratio Test**, **Deviance Goodness-of-Fit Test** and **Pearson's Chi-Squared Test** respectively.

#### **Overall Model Significance** (Likelihood-Ratio Test)
$p$-value being essentialy $0$ shows that our risk factors actually have predictive power.

In [20]:
dif.dev = fit.count.train$null.deviance - fit.count.train$deviance
dif.df = fit.count.train$df.null - fit.count.train$df.residual
p_val_lrt = pchisq(dif.dev, df = dif.df, lower.tail = FALSE)
dif.dev; dif.df; p_val_lrt

[1] 8626.824

[1] 16

[1] 0

#### **Model Specification** (Deviance Goodness-of-Fit Test)
$p$-value being approximately 1 shows that Poisson GLM is well specified.

In [21]:
dev = fit.count.train$deviance
df_res = fit.count.train$df.residual
p_val_gof = pchisq(dev, df = df_res, lower.tail = FALSE)
dev; p_val_gof

[1] 1052743

[1] 1

#### **Overdispersion** (Pearson's Chi-Squared Test)
The large value of X2 shows that sum of squared standardised errors is large, and thus we conclude that the overdispersion is present. 

In [22]:
fit.val = predict(fit.count.train, newdata = df_train, type = 'response')
X2 = sum((fit.val-df_train$num_cl)^2/fit.val) #sum of squared standardised errors
df_train_residual = fit.count.train$df.residual
p_val_X2 = pchisq(X2, df = df_train_residual, lower.tail = FALSE)
est.phi = X2/df_train_residual #estimate of dispersion parameter
X2; p_val_X2; est.phi

[1] 3376109

[1] 0

[1] 2.139104

## ***Modeling Claim Severity $Y_n$***
In the following `code` cells we first 